In [ ]:
# Exploratory factor analysis

# This notebook performs an exploratory factor analysis (EFA) of the mean ratings across the experiential dimensions. 
# The analysis first assesses the suitability of the data for factor analysis 
# using Bartlett’s test of sphericity and the Kaiser–Meyer–Olkin (KMO) measure.
# The number of factors is then evaluated using a scree plot, 
# followed by extraction of a three-factor solution using maximum likelihood estimation with promax rotation. 
# Factor loadings and explained variance are examined, and factor scores are subsequently calculated for each word
# to create a reduced three-factor dataset.

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

import pandas as pd
from factor_analyzer import FactorAnalyzer
import matplotlib.pyplot as plt
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
from factor_analyzer.factor_analyzer import calculate_kmo


# ============================================================
# 2. Load dataset
# ============================================================

df = pd.read_csv('../materials/MultiNorms_GER_summary_level_dataset.csv',  index_col=0)
# Format dataframe for factor analysis, include only mean columns 
mean_columns = [col for col in df.columns[0:68] if '_mean' in col]  # select only mean columns
df = df[mean_columns]
# Drop valence signed mean, take only the absolute mean
df = df.drop(['valence_mean'], axis=1)

In [ ]:
# ============================================================
# 3. Adequacy test
# ============================================================

## Bartlett’s test of sphericity checks whether or not the observed variables intercorrelate at all using the observed correlation matrix against the identity matrix. 
## If the test found statistically insignificant, you should not employ a factor analysis.

chi_square_value,p_value=calculate_bartlett_sphericity(df)
print("Bartlett’s test of sphericity ",chi_square_value, p_value)


## Kaiser-Meyer-Olkin (KMO) Test measures the suitability of data for factor analysis. It determines the adequacy for each observed variable and for the complete model. 
## KMO values range between 0 and 1. Value of KMO less than 0.6 is considered inadequate.

kmo_all,kmo_model=calculate_kmo(df)
print("KMO: ", kmo_model)
      

In [ ]:
# ============================================================
# 4. Perform factor analysis 
# ============================================================

n_variables = len(df.columns)
factor_model = FactorAnalyzer()
factor_model.fit(df)
eigenvalues, _ = factor_model.get_eigenvalues() 

plt.scatter(range(1,df.shape[1]+1),eigenvalues)
plt.plot(range(1,df.shape[1]+1),eigenvalues)
plt.title('Scree Plot')
plt.xlabel('Factors')
plt.ylabel('Eigenvalue')
plt.grid()
plt.show()

In [ ]:
# ============================================================
# 5. Extract 3 factors based on Scree plot 
# ============================================================

factor_model = FactorAnalyzer(n_factors=3, method='ml', rotation='promax')#, is_corr_matrix=False) 
factor_model.fit(df)

## Variance explained
# 1. Sum of squared loadings (variance)
# 2. Proportional variance
# 3. Cumulative variance

factor_model.get_factor_variance()


In [ ]:
# ============================================================
# 5. Create dataframe with factor loadings
# ============================================================

factor_df= pd.DataFrame(factor_model.loadings_, index=df.columns)
factor_df

In [ ]:
# ============================================================
# 6. Create reduced dataset 
# ============================================================

reduced_df = factor_model.transform(df)
reduced_df=pd.DataFrame(reduced_df, index=df.index, columns=['F1', 'F2', 'F3'])
reduced_df.to_csv("MultiNorms_GER_reduced_df_efa_3factors.csv", encoding='utf-8-sig')